# Uydu Görüntülerinden Görüntü İşleme Yöntemleri ile Oraj Tespiti
### TÜBİTAK 2209-A — Akif Karaca | Danışman: Dr. Öğr. Üyesi Gülşah Karaduman

---

**Çalıştırma sırası:** Hücreleri yukarıdan aşağıya sırayla çalıştırın. Her hücrenin üstünde ne yaptığı yazılıdır.

**Önce yapılacak:** `Runtime → Change runtime type → T4 GPU` seçin.

**Toplam süre:** Veri indirme ~40 dk, her model eğitimi ~40 dk, geri kalanı ~1 saat.

## Hücre 1 — Ortam kontrolü ve Google Drive bağlantısı

GPU'nun seçili olduğunu doğrular ve Drive'ı bağlar.
Drive önemlidir: Colab oturumu kapandığında `/content` silinir, eğittiğiniz modeller kaybolur.
Sonuçları Drive'a kopyalayarak bu riski ortadan kaldırıyoruz.

In [ ]:
import tensorflow as tf

gpu = tf.config.list_physical_devices('GPU')
print("TensorFlow:", tf.__version__)
print("GPU:", gpu if gpu else "YOK — Runtime > Change runtime type > T4 GPU seçin!")

from google.colab import drive
drive.mount('/content/drive')

import os
KAYIT = '/content/drive/MyDrive/oraj_2209a'
os.makedirs(KAYIT, exist_ok=True)
print("Sonuçlar buraya kopyalanacak:", KAYIT)

## Hücre 2 — Gerekli kütüphanelerin kurulumu

Colab'da TensorFlow, numpy, pandas, h5py, OpenCV zaten yüklüdür — onlara dokunmuyoruz.
Sadece eksik olan üçünü kuruyoruz.

**Uyarı:** `pip install tensorflow` ÇALIŞTIRMAYIN, Colab'ın ortamını bozar.

In [ ]:
!pip install -q boto3 gradio scipy
print("Kurulum tamam.")

## Hücre 3 — Proje dosyalarının yüklenmesi

Üç Python dosyasını yükleyin:

| Dosya | İçeriği |
|---|---|
| `sevir_pipeline.py` | Veri hattı, modeller, eğitim, değerlendirme |
| `ek_calismalar.py` | Optik akış analizi (İP-2) + Gradio arayüzü |
| `lght_dogrulama.py` | Yıldırım verisiyle etiket doğrulama (opsiyonel) |

Aşağıdaki hücre bir dosya seçme penceresi açar. Üçünü birden seçebilirsiniz.

*Alternatif:* Sol paneldeki klasör ikonuna tıklayıp dosyaları sürükleyip bırakabilirsiniz.

In [ ]:
from google.colab import files
yuklenen = files.upload()
print("\nYüklenen dosyalar:", list(yuklenen.keys()))

## Hücre 4 — SEVIR veri setinin indirilmesi

SEVIR (Storm EVent ImagRy), MIT Lincoln Laboratory tarafından hazırlanmış,
NOAA Storm Events veri tabanıyla eşleştirilmiş açık erişimli bir uydu görüntü arşividir.

**Sadece IR107 (termal kızılötesi) kanalını indiriyoruz.** VIS kanalı gerekmiyor çünkü:
- IR 192×192, VIS 768×768 → IR onlarca kat daha küçük
- IR gece de çalışır, VIS çalışmaz
- Derin konveksiyonun imzası zaten bulut tepesi sıcaklığındadır

İndirilen dosyalar:
- `CATALOG.csv` — olay kataloğu (kimlik, tarih, dosya, indeks)
- `SEVIR_IR107_STORMEVENTS_2018_0101_0630.h5` — oraj olayları (Ocak–Haziran)
- `SEVIR_IR107_RANDOMEVENTS_2018_0501_0831.h5` — rastgele olaylar (Mayıs–Ağustos)

⏱️ **Bu hücre 30–60 dakika sürer.** Sekmeyi açık tutun.

In [ ]:
import boto3, os
from botocore import UNSIGNED
from botocore.config import Config
from tqdm import tqdm

DATA_DIR = 'data/sevir'
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

hedefler = [
    'CATALOG.csv',
    'data/ir107/2018/SEVIR_IR107_STORMEVENTS_2018_0101_0630.h5',
    'data/ir107/2018/SEVIR_IR107_RANDOMEVENTS_2018_0501_0831.h5',
]

for key in hedefler:
    yerel = os.path.join(DATA_DIR, key.replace('data/', '', 1))
    if os.path.exists(yerel):
        print('mevcut, atlanıyor:', yerel)
        continue
    os.makedirs(os.path.dirname(yerel), exist_ok=True)
    boyut = s3.head_object(Bucket='sevir', Key=key)['ContentLength']
    print(f"{key}  ({boyut/1e9:.2f} GB)")
    with tqdm(total=boyut, unit='B', unit_scale=True) as bar:
        s3.download_file('sevir', key, yerel, Callback=bar.update)

print("\nİndirme tamamlandı.")
!df -h /content | tail -1

## Hücre 5 — Veri indeksinin kurulması

Bu adım projenin **en kritik metodolojik aşamasıdır.** `build_index()` şunları yapar:

1. **Kanal filtresi** — sadece IR107 satırları alınır, böylece VIS/IR indeks karışıklığı yapısal olarak imkânsız hale gelir
2. **Disk kontrolü** — katalogda olup diskte olmayan dosyalar elenir
3. **Ortak zaman penceresi** — her iki sınıf da 1 Mayıs – 30 Haziran 2018 aralığına kısıtlanır

Üçüncü madde neden önemli? Oraj dosyası Ocak–Haziran, rastgele dosya Mayıs–Ağustos kapsıyor.
Filtre uygulanmazsa oraj sınıfı kışı da içerirken karşıt sınıf sadece yaz olur.
Termal görüntüde kış/yaz farkı devasadır — model oraj yerine **mevsimi** öğrenir.
Buna literatürde *kısayol öğrenmesi* (shortcut learning) denir.

4. **Geçerlilik denetimi** — katalogdaki indeksler dosya boyutuyla karşılaştırılır, uyumsuzlar atılır

In [ ]:
from sevir_pipeline import *

idx = build_index(time_start='2018-05-01', time_end='2018-07-01')

## Hücre 6 — Veri kalitesinin görsel kontrolü

`diagnose()` her sınıftan örnek kareler çizer ve ham/normalize değer aralıklarını yazdırır.

**Neye bakmalısınız:**
- Üst sıra (ORAJ): parlak, dokulu, keskin kenarlı yapılar → soğuk konvektif bulut tepeleri
- Alt sıra (NORMAL): daha düz, sönük görüntüler

İki sıra birbirinden ayırt edilemiyorsa normalizasyonda sorun var demektir.

`zamansal_dagilim()` ise iki sınıfın yıl içindeki dağılımını karşılaştırır.
**Histogramlar örtüşüyorsa mevsimsel yanlılık kontrol altında demektir** —
bu, rapora konacak bir kanıt figürüdür.

In [ ]:
diagnose(idx)
zamansal_dagilim(idx)

## Hücre 7 — Olay bazlı eğitim/doğrulama bölmesi

Aynı olaya ait ardışık kareler 5 dakika arayla kaydedildiği için birbirleriyle **çok yüksek korelasyona** sahiptir.

Bölme kare düzeyinde yapılırsa aynı fırtınanın farklı anları hem eğitim hem doğrulama kümesine dağılır;
model doğrulama sırasında pratikte daha önce gördüğü sahnelerle karşılaşır ve performans **yapay olarak** yükselir.
Buna *veri sızıntısı* (data leakage) denir.

Bu yüzden bölme **olay kimliği (event id) düzeyinde** yapılır: bir olay eğitime giderse tüm kareleri eğitimde kalır.

Bölme `split_event_level.json` dosyasına kaydedilir. Hem eğitim hem değerlendirme aynı dosyayı okur —
böylece iki modelin karşılaştırılması adil olur.

In [ ]:
split = make_split(idx)
train_gen, val_gen = make_generators(idx, split, batch_size=16, frames_per_event=9)
cw = class_weights(idx, split)

## Hücre 8 — ResNet50 modelinin eğitimi

**İki aşamalı eğitim stratejisi kullanılıyor:**

| Aşama | Ne yapılır | Öğrenme oranı |
|---|---|---|
| 1 | Gövde tamamen donuk, sadece yeni sınıflandırıcı eğitilir | 1×10⁻³ |
| 2 | Gövdenin üst blokları açılır, **BatchNorm katmanları donuk kalır** | 2×10⁻⁵ |

BatchNorm'un donuk tutulması kritiktir. İlk denemelerde gövdenin tamamı bir anda açıldığında
model çökmüştü: `val_loss` 18,8'e fırladı ve tüm örneklere "oraj" demeye başladı.
Kademeli çözme + BN dondurma bu sorunu tamamen çözdü.

⏱️ **~40 dakika sürer.**

In [ ]:
!rm -f resnet_history.csv   # eski kayıt varsa grafikleri bozmasın

model_resnet = build_model('resnet')
train_stable(model_resnet, train_gen, val_gen, name='resnet', cw=cw,
             epochs_head=8, epochs_ft=25, unfreeze_from='conv4')

## Hücre 9 — DenseNet121 modelinin eğitimi

Araştırma önerisindeki **İP-3 taahhüdü**: birden fazla mimarinin aynı protokol altında karşılaştırılması.

DenseNet, katmanlar arası yoğun bağlantılarla bilgi akışını güçlendirir ve
daha az parametreyle benzer temsil kapasitesine ulaşır.

Aynı bölme, aynı jeneratörler, aynı eğitim stratejisi kullanılıyor — karşılaştırma adil olsun diye.

⏱️ **~40 dakika sürer.**

In [ ]:
!rm -f densenet_history.csv

model_densenet = build_model('densenet')
train_stable(model_densenet, train_gen, val_gen, name='densenet', cw=cw,
             epochs_head=8, epochs_ft=25, unfreeze_from='conv4')

## Hücre 10 — Model performansının değerlendirilmesi

Her iki model, eğitimde **hiç görülmemiş** olaylardan oluşan doğrulama kümesinde test edilir.

Üretilen çıktılar:
- Sınıflandırma raporu (kesinlik, duyarlılık, F1 — sınıf bazında)
- ROC-AUC skoru
- Karışıklık matrisi (her model için ayrı PNG)

**Sayıları not alın** — rapordaki Tablo 3'e bunlar girecek.

In [ ]:
print("="*60, "\nResNet50\n", "="*60)
yt_r, yp_r = evaluate('resnet_best.keras', idx, split)

print("\n" + "="*60, "\nDenseNet121\n", "="*60)
yt_d, yp_d = evaluate('densenet_best.keras', idx, split)

## Hücre 11 — Rapor için grafiklerin üretilmesi

Üç görsel üretilir:

1. **Eğitim eğrileri** — doğruluk ve kayıp, epoch bazında. Eğitim/doğrulama eğrileri
   birbirine yakınsa aşırı öğrenme yok demektir.
2. **ROC eğrisi** — iki model tek grafikte karşılaştırmalı
3. **Yanlış alarm örnekleri** — "normal" etiketli ama model yüksek güvenle "oraj" dediği kareler

Üçüncüsü raporun en güçlü argümanıdır: bu görüntülerde gerçekten konvektif yapı görünüyorsa,
yanlış alarmların bir kısmı modelin hatası değil **etiket gürültüsüdür.**
SEVIR'in RANDOMEVENTS sınıfı "hava olayı yok" demek değil — rastgele seçilmiş yaz kareleri,
bir kısmında konveksiyon var.

In [ ]:
plot_history('resnet_history.csv')
plot_history('densenet_history.csv')

plot_roc({'ResNet50': (yt_r, yp_r), 'DenseNet121': (yt_d, yp_d)})

yanlis_alarm_gorsel(idx, split, yt_r, yp_r, n=6, esik=0.8)

## Hücre 12 — İP-2: Optik akış ile bulut hareketlerinin çıkarılması

Araştırma önerisindeki **İP-2 taahhüdü.** Ardışık uydu görüntüleri arasındaki hareket alanı
Farnebäck optik akış algoritmasıyla hesaplanır.

İki çıktı üretilir:

1. **Niteliksel görsel** — t, t+5 dk ve akış vektörleri (oraj ve normal için ayrı)
2. **Nicel analiz** — her olaydan dört öznitelik çıkarılıp iki sınıf istatistiksel olarak karşılaştırılır:
   - ortalama hız, 95. yüzdelik hız
   - bulutlu bölge hızı
   - **yayılma (diverjans)** — konvektif örs yayılması pozitif diverjans üretir, fiziksel bir imzadır

Mann-Whitney U testi ile anlamlılık kontrol edilir.

⚠️ **Dikkat:** Bu analiz betimleyicidir; %90 doğruluk iddiası YAPMAYIN.
Raporda "öznitelikler çıkarıldı ve karşılaştırıldı, ancak sınıflandırma modeline entegre edilmedi" deyin.

⏱️ ~15 dakika.

In [ ]:
from ek_calismalar import *

optik_akis_gorsel(idx)
df_akis = optik_akis_analizi(idx, n=25)

## Hücre 13 — Kullanıcı dostu arayüz (Gradio demosu)

Araştırma önerisindeki **"sonuçların kullanıcı dostu bir arayüzle sunulması"** hedefi.

Arayüz doğrulama kümesinden örnek çeker, modelin olasılık çıktısını ve
tahminin doğru olup olmadığını gösterir.

Hücre çalıştığında **paylaşılabilir bir bağlantı** üretilir (72 saat geçerli).

📸 **Ekran görüntüsü alın** — rapora eklenecek.

*Not:* Bu hücre siz durdurana kadar çalışmaya devam eder. Devam etmek için hücreyi durdurun.

In [ ]:
arayuz_baslat('resnet_best.keras', idx)

## Hücre 14 — (Opsiyonel) Yıldırım verisiyle etiket kalitesinin doğrulanması

**Bu adım etiketleri DEĞİŞTİRMEZ** — mevcut sonuçlarınız geçerli kalır.
Yalnızca STORMEVENTS/RANDOMEVENTS ayrımının meteorolojik olarak ne kadar tutarlı olduğunu ölçer.

Mantık: meteorolojik tanım gereği **oraj, yıldırım üreten konvektif fırtınadır.**
Dolayısıyla yıldırım sayımı, etiket kalitesi için doğrudan bir ölçüttür.

Raporda şu anda bir *yorum* olan cümle ("yanlış alarmlar etiket gürültüsünden kaynaklanıyor olabilir"),
bu analizle *ölçülmüş bir bulguya* dönüşür.

⚠️ `lght_kesif()` çıktısını inceleyin. Beklenen yapı: her anahtar bir olay kimliği,
değeri `(N, 5)` boyutunda sönme listesi. Farklıysa `_sayim_cikar()` uyarlanmalıdır.

**Karar kuralı:** Format 30 dakikada çözülmezse bu adımı bırakın. Bonus çalışmadır.

In [ ]:
from lght_dogrulama import *

lght_indir()
lght_kesif()        # ÖNCE bu — format çıktısını inceleyin

In [ ]:
# Format doğrulandıktan sonra çalıştırın
df_lght = lght_dogrulama(idx, n=60)

## Hücre 15 — Tüm çıktıların Google Drive'a kopyalanması

**Bu hücreyi mutlaka çalıştırın.** Colab oturumu kapandığında `/content` silinir;
eğitilmiş modeller, grafikler ve bölme dosyası kaybolur.

Kopyalananlar:
- `.keras` — eğitilmiş model ağırlıkları
- `.csv` — eğitim geçmişi
- `.png` — rapora girecek tüm grafikler
- `split_event_level.json` — bölme bilgisi (tekrarlanabilirlik için)

In [ ]:
!cp -v *.keras *.csv *.png split_event_level.json {KAYIT}/ 2>/dev/null

print("\n--- Drive'daki dosyalar ---")
!ls -lh {KAYIT}/

## Hücre 16 — Rapora girecek sayıların özeti

Bu hücre, sonuç raporundaki tabloları doldurmak için gereken tüm sayıları tek yerde toplar.
Çıktıyı kopyalayıp saklayın.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

print("="*64)
print("RAPOR İÇİN ÖZET")
print("="*64)

n1, n0 = int((idx.label==1).sum()), int((idx.label==0).sum())
print(f"\nVERİ SETİ")
print(f"  Toplam olay      : {len(idx)}")
print(f"  Oraj (1)         : {n1}")
print(f"  Normal (0)       : {n0}")
print(f"  Eğitim olayı     : {len(split['train'])}")
print(f"  Doğrulama olayı  : {len(split['val'])}")
print(f"  Eğitim görüntüsü : {len(train_gen.samples)}")
print(f"  Doğrulama görüntüsü: {len(val_gen.samples)}")

tr = idx[idx['id'].isin(set(split['train']))]
va = idx[idx['id'].isin(set(split['val']))]
print(f"\n  Eğitim   -> Oraj: {int((tr.label==1).sum())}  Normal: {int((tr.label==0).sum())}")
print(f"  Doğrulama-> Oraj: {int((va.label==1).sum())}  Normal: {int((va.label==0).sum())}")

for ad, yt, yp in [('ResNet50', yt_r, yp_r), ('DenseNet121', yt_d, yp_d)]:
    print(f"\n{'-'*64}\n{ad}\n{'-'*64}")
    print(classification_report(yt, (yp[:len(yt)]>0.5).astype(int),
                                target_names=['Normal (0)','Oraj (1)'], digits=3))
    print(f"ROC AUC: {roc_auc_score(yt, yp[:len(yt)]):.4f}")

print("\n" + "="*64)
print("Bu çıktıyı kopyalayıp rapor tablolarına girin.")
print("="*64)